In [1]:
import re
# !pip install sentencepiece
import sentencepiece as spm
import pandas as pd
import numpy as np
import itertools

In [2]:
df = pd.read_csv("hf://datasets/core-outline/llama-2-7b-chat-hf/blogdata.csv")

In [3]:
# df['ans'] = df['ans']+ '<|endoftext|>'

text_arr = df['ans'].tolist()

In [4]:
# text = " ".join(df['ans'].tolist())

In [5]:
# len(np.unique(text.split(" ")))

In [6]:
def pretokenize(text):
    pattern = re.compile(r'([ A-Za-z]+|\d+|[^A-Za-z0-9]+)')
    chunks = pattern.findall(text)
    return chunks

In [7]:
def byte_tokenize(text):
    tokens = []
    for chunk in text:
      for byte in chunk.encode('utf-8'):
        tokens.append(byte)
    return tokens

In [8]:
def tokenize_text(text):
    pretokenized_chunks = pretokenize(text)
    byte_tokens = byte_tokenize(pretokenized_chunks)
    return byte_tokens

In [9]:
tokens = tokenize_text(df['ans'][0])
# print(f"Original text: {text}")
# print(f"Tokens: {tokens}")
print(f"Number of tokens: {len(tokens)}")

Number of tokens: 7382


In [10]:
df['tokenized'] = df['ans'].apply(lambda x: tokenize_text(x))

In [11]:
concatenated_tokens = [ i for i in itertools.chain.from_iterable(df['tokenized'].tolist()) ]

In [12]:
concatenated_tokens[-1]

46

In [13]:
start = 0
stop = len(concatenated_tokens)
# stop = 10000
step = 2048

In [14]:
chunked_texts = []

In [15]:
for i in np.arange(start, stop, step):
    chunked_texts.append(concatenated_tokens[i:i+2048])

In [16]:
len(chunked_texts)

2463

In [17]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, LayerNormalization, Embedding, Input, MultiHeadAttention
from tensorflow.keras.models import Sequential, Model


In [18]:
vocab_size = 48397 
max_seq_len = 2048
num_layers = 2
embed_dim = 7680
num_heads = 40
ff_dim = 15360
epochs = 2

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math
from tqdm import tqdm
from colorama import Fore, Style, init

In [20]:
init(autoreset=True)


In [21]:
def create_vocab(text):
    words = text.split()
    unique_words = set(words)
    vocab = {word: i+4 for i, word in enumerate(unique_words)}
    vocab['<pad>'] = 0
    vocab['<unk>'] = 1
    vocab['<sos>'] = 2
    vocab['<eos>'] = 3
    return vocab

In [22]:
vocabulary = create_vocab("".join(text_arr))

In [23]:
class TextDataset(Dataset):
    def __init__(self, vocab=None):
        text = "".join(text_arr)

        if vocab is None:
            self.vocab = create_vocab(text)
        else:
            self.vocab = vocab
        
        self.data = [self.vocab.get(word, self.vocab['<unk>']) for word in text.split()]
        self.data += [self.vocab['<eos>']] * max_seq_len
        
    def __len__(self):
        return len(self.data) - max_seq_len + 1
    def __getitem__(self, idx):
        sequence = self.data[idx:idx+max_seq_len]
        input_sequence = torch.tensor(sequence[:-1], dtype=torch.long)
        target_sequence = torch.tensor(sequence[1:], dtype=torch.long)
        return input_sequence, target_sequence

In [24]:
data = TextDataset(vocabulary)
dataloader = DataLoader(data, batch_size=4,
                        shuffle=True, num_workers=4)

In [25]:
class TransformerModel(nn.Module):
    def __init__(self, vocab_size):
        super(TransformerModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, EMBEDDING_SIZE)
        self.pos_encoder = PositionalEncoding(EMBEDDING_SIZE, MAX_SEQ_LENGTH)
        self.transformer_decoder_layer = nn.TransformerDecoderLayer(
            d_model=EMBEDDING_SIZE, nhead=NHEAD, dim_feedforward=FFN_HID_DIM
        )
        self.transformer_decoder = nn.TransformerDecoder(
            self.transformer_decoder_layer, num_layers=NUM_DECODER_LAYERS
        )
        self.fc_out = nn.Linear(EMBEDDING_SIZE, vocab_size)

    def generate_square_subsequent_mask(self, sz):
        mask = torch.triu(torch.ones(sz, sz) * float('-inf'), diagonal=1)
        return mask
    def forward(self, src):
        src_mask = self.generate_square_subsequent_mask(src.size(0)).to(src.device)
        src = self.embedding(src) * math.sqrt(EMBEDDING_SIZE)
        src = self.pos_encoder(src)
        output = self.transformer_decoder(tgt=src, memory=src, tgt_mask=src_mask)
        output = self.fc_out(output)
        return output

In [28]:
data

In [26]:
# progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f'Epoch {epoch+1}/{epochs}', leave=True)

In [27]:
# for epoch in range(2):
#     total_loss = 0
#     progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f'Epoch {epoch+1}/{epochs}', leave=True)
#     for i, (src, tgt) in progress_bar:
#         src = src.transpose(0, 1)
#         tgt_output = tgt.transpose(0, 1)
#         optimizer.zero_grad()
#         output = model(src)
#         loss = criterion(output.view(-1, len(dataset.vocab)), tgt_output.reshape(-1))
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()

#         # Color-coded logging
#         if loss.item() < 1.0:
#             color = Fore.GREEN
#         elif loss.item() < 2.0:
#             color = Fore.YELLOW
#         else:
#             color = Fore.RED
#         progress_bar.set_postfix(loss=f"{color}{loss.item():.4f}{Style.RESET_ALL}")
#     avg_loss = total_loss / len(dataloader)
#     print(f'End of Epoch {epoch+1}, Average Loss: {avg_loss:.4f}')